# 03 - State Filtering (Heteroskedastic Kalman)

Goal: recover latent efficient log-odds $\hat x_t$ from noisy observations.

Model:
$$
x_t = x_{t-1} + \eta_t, \quad \eta_t \sim \mathcal N(0, q_t)
$$
$$
y_t = x_t + \varepsilon_t, \quad \varepsilon_t \sim \mathcal N(0, r_t), \quad r_t = \sigma^2_{\varepsilon,t}
$$
with adaptive process variance $q_t$ from recent latent innovations.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
inp = Path('stage2_logit_noise.csv')
if not inp.exists():
    raise FileNotFoundError('Run notebook 02 first to generate stage2_logit_noise.csv')

df = pd.read_csv(inp)
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
df = df.sort_values('timestamp').reset_index(drop=True)
df.head()

## Scalar heteroskedastic Kalman filter

In [ ]:
def run_kalman(y, r, q_floor=1e-8):
    n = len(y)
    x_filt = np.zeros(n)
    p_filt = np.zeros(n)
    innov = np.zeros(n)
    k_gain = np.zeros(n)

    x_prev = y[0]
    p_prev = np.var(y[: min(100, n)]) + 1e-4

    for t in range(n):
        q_t = max(q_floor, np.var(np.diff(x_filt[max(1, t - 60):t + 1])) if t > 2 else 1e-4)

        x_pred = x_prev
        p_pred = p_prev + q_t

        innov_t = y[t] - x_pred
        s_t = p_pred + max(r[t], 1e-8)
        k_t = p_pred / s_t

        x_new = x_pred + k_t * innov_t
        p_new = (1 - k_t) * p_pred

        x_filt[t] = x_new
        p_filt[t] = p_new
        innov[t] = innov_t
        k_gain[t] = k_t

        x_prev, p_prev = x_new, p_new

    return x_filt, p_filt, innov, k_gain

y = df['y_logit'].to_numpy(dtype=float)
r = df['sigma2_eps'].to_numpy(dtype=float)
x_hat, p_hat, innov, k_gain = run_kalman(y, r)

df['x_hat'] = x_hat
df['kalman_var'] = p_hat
df['innovation'] = innov
df['kalman_gain'] = k_gain

df[['timestamp', 'y_logit', 'x_hat', 'innovation']].head()

## Innovation diagnostics

In [ ]:
diag = {
    'innovation_mean': float(df['innovation'].mean()),
    'innovation_std': float(df['innovation'].std()),
    'innovation_abs_p95': float(df['innovation'].abs().quantile(0.95)),
}
diag

In [ ]:
out_cols = [
    'timestamp', 'p_clipped', 'y_logit', 'sigma2_eps',
    'x_hat', 'kalman_var', 'innovation', 'kalman_gain',
]
df[out_cols].to_csv('stage3_kalman.csv', index=False)
print('saved:', Path('stage3_kalman.csv').resolve())
print('rows:', len(df))